# E1: LSTM chỉ dùng forcing khí tượng (CAMELS trong Caravan)

Thí nghiệm này tương ứng với model **"Global LSTM without static features"** trong Kratzert et al. (2019, WRR), có NSE trung vị 0.63 trên 531 lưu vực CAMELS.

| Thiết lập | Giá trị |
|---|---|
| Đầu vào | 22 biến khí tượng ERA5-Land, **không** có thuộc tính tĩnh |
| Đầu ra | `streamflow` (mm/ngày) của ngày cuối cửa sổ |
| Cửa sổ | 365 ngày, stride 1 |
| Train / Val / Test | 01/10/1980–30/09/1990 / 01/10/1990–30/09/1995 / 01/10/1995–30/09/2010 |
| Model | LSTM 256 cell, dropout 0.4, 1 lớp Linear ở cuối |
| Loss | NSE* (trung bình NSE có trọng số theo từng lưu vực) |

**Các bước trong notebook**

1. Cấu hình chung (**bạn chỉ cần sửa ô này**)
2. Kiểm tra môi trường
3. Chuyển CSV → netCDF (chỉ chạy 1 lần)
4. Kiểm tra chất lượng và chọn lưu vực
5. Tạo file cấu hình NeuralHydrology
6. Train
7. Xem quá trình train, chọn epoch tốt nhất theo tập validation
8. Đánh giá trên tập test
9. Xem hydrograph của một số lưu vực

**Cách dùng:** lần đầu để `QUICK_TEST = True` để chạy thử nhanh (20 lưu vực, 2 epoch, LSTM nhỏ), kiểm tra mọi bước không lỗi. Sau đó đổi `QUICK_TEST = False`, chạy lại từ ô 1 để train thật. Bước 3 không cần chạy lại.

## 0. Cài đặt (chạy trong terminal, chỉ 1 lần)

```bash
conda create -n hydro python=3.11 -y
conda activate hydro
pip install torch --index-url https://download.pytorch.org/whl/cu128   # RTX 50xx cần CUDA >= 12.8
pip install neuralhydrology jupyter
```

Lưu ý: đường dẫn tới thư mục dữ liệu **không nên chứa dấu tiếng Việt hoặc khoảng trắng**, vì thư viện netCDF trên Windows hay lỗi với các ký tự này.

## 1. Cấu hình chung

In [ ]:
from pathlib import Path

# ===================== SỬA Ở ĐÂY =====================
CSV_DIR = Path(r"data\camels")            # thư mục chứa các file camels_*.csv
CARAVAN_ROOT = Path(r"data\caravan")      # nơi ghi netCDF theo cấu trúc Caravan (timeseries/netcdf/camels/...)
QUICK_TEST = True                         # True: chạy thử nhanh; False: train thật
# =====================================================

RESULT_DIR = Path("result")
SUBSET = "camels"
SEED = 42

# 22 biến khí tượng (đã bỏ surface_net_solar_radiation_min và potential_evaporation_sum_ERA5_LAND)
DYNAMIC_INPUTS = [
    "total_precipitation_sum",
    "temperature_2m_min", "temperature_2m_mean", "temperature_2m_max",
    "dewpoint_temperature_2m_min", "dewpoint_temperature_2m_mean", "dewpoint_temperature_2m_max",
    "surface_net_solar_radiation_mean", "surface_net_solar_radiation_max",
    "surface_net_thermal_radiation_min", "surface_net_thermal_radiation_mean", "surface_net_thermal_radiation_max",
    "surface_pressure_min", "surface_pressure_mean", "surface_pressure_max",
    "u_component_of_wind_10m_min", "u_component_of_wind_10m_mean", "u_component_of_wind_10m_max",
    "v_component_of_wind_10m_min", "v_component_of_wind_10m_mean", "v_component_of_wind_10m_max",
    "potential_evaporation_sum_FAO_PENMAN_MONTEITH",
]
TARGET = "streamflow"

# Mốc chia A (năm thủy văn: 1/10 -> 30/9)
PERIODS = {
    "train":      ("1980-10-01", "1990-09-30"),
    "validation": ("1990-10-01", "1995-09-30"),
    "test":       ("1995-10-01", "2010-09-30"),
}
SEQ_LENGTH = 365
MIN_VALID_YEARS = {"train": 5, "validation": 2, "test": 3}   # số năm có nhãn tối thiểu để giữ lưu vực

# Siêu tham số
if QUICK_TEST:
    EXPERIMENT = "e1_lstm_forcing_only_QUICK"
    N_BASINS_QUICK = 20
    HIDDEN_SIZE, EPOCHS = 32, 2
    LEARNING_RATE = {0: 1e-3}
else:
    EXPERIMENT = "e1_lstm_forcing_only"
    HIDDEN_SIZE, EPOCHS = 256, 30
    LEARNING_RATE = {0: 1e-3, 10: 5e-4, 25: 1e-4}
BATCH_SIZE = 256
OUTPUT_DROPOUT = 0.4

NC_DIR = CARAVAN_ROOT / "timeseries" / "netcdf" / SUBSET
E1_DIR = RESULT_DIR / "e1"
E1_DIR.mkdir(parents=True, exist_ok=True)

print("CSV:", CSV_DIR, "| ton tai:", CSV_DIR.is_dir())
print("Ket qua se luu tai:", E1_DIR.resolve())
if QUICK_TEST:
    print("\n*** QUICK_TEST = True: day chi la lan chay thu, KHONG phai ket qua that ***")

## 2. Kiểm tra môi trường

In [ ]:
import sys, platform, importlib
import numpy as np
import pandas as pd
import torch

print("OS:", platform.platform(), "| Python:", sys.version.split()[0])
for pkg in ["torch", "neuralhydrology", "pandas", "numpy", "xarray", "netCDF4"]:
    m = importlib.import_module(pkg)
    print(f"  {pkg}: {getattr(m, '__version__', 'installed')}")

if torch.cuda.is_available():
    DEVICE = "cuda:0"
    x = torch.randn(1024, 1024, device=DEVICE); _ = (x @ x).sum().item()
    print("GPU:", torch.cuda.get_device_name(0), "| CUDA", torch.version.cuda, "-> OK")
else:
    DEVICE = "cpu"
    print("KHONG thay GPU, se chay tren CPU (rat cham). Kiem tra lai ban PyTorch.")

## 3. Chuyển CSV → netCDF (chỉ cần chạy 1 lần)

Bộ đọc Caravan của NeuralHydrology đọc file tại `timeseries/netcdf/camels/<id>.nc`. Ô này đọc từng file CSV, kiểm tra chất lượng lưu lượng, rồi ghi ra netCDF (nén, float32). **File CSV gốc không bị thay đổi.**

Kiểm tra chất lượng khi chuyển: giá trị lưu lượng **âm** là lỗi đo, được đổi thành NaN. Lưu lượng **không được nội suy**.

File đã chuyển rồi sẽ được bỏ qua. Muốn chuyển lại thì đặt `OVERWRITE = True`.

In [ ]:
import xarray as xr
from tqdm import tqdm

OVERWRITE = False

def csv_to_netcdf(csv_path: Path, nc_path: Path) -> dict:
    df = pd.read_csv(csv_path, parse_dates=["date"]).set_index("date").sort_index()
    missing = [c for c in DYNAMIC_INPUTS + [TARGET] if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path.name} thieu cot: {missing}")
    n_neg = int((df[TARGET] < 0).sum())
    df.loc[df[TARGET] < 0, TARGET] = np.nan
    ds = xr.Dataset.from_dataframe(df.astype("float32"))
    enc = {v: {"zlib": True, "complevel": 4} for v in ds.data_vars}
    nc_path.parent.mkdir(parents=True, exist_ok=True)
    ds.to_netcdf(nc_path, encoding=enc)
    return {"gauge_id": csv_path.stem, "n_negative_set_nan": n_neg}

csv_files = sorted(CSV_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"Khong tim thay file CSV nao trong {CSV_DIR.resolve()}. Kiem tra lai CSV_DIR o o 1.")
print(f"Tim thay {len(csv_files)} file CSV")
log = []
for f in tqdm(csv_files, desc="CSV -> netCDF"):
    out = NC_DIR / f"{f.stem}.nc"
    if out.exists() and not OVERWRITE:
        continue
    log.append(csv_to_netcdf(f, out))

if log:
    log = pd.DataFrame(log)
    log.to_csv(E1_DIR / "conversion_log.csv", index=False)
    print(f"Da chuyen {len(log)} file. So luu vuc co gia tri am: {(log.n_negative_set_nan > 0).sum()}")
print("So file netCDF hien co:", len(list(NC_DIR.glob('*.nc'))))

# Kiểm tra: đọc lại 1 file bằng chính hàm của NeuralHydrology và so với CSV
from neuralhydrology.datasetzoo.caravan import load_caravan_timeseries
check_id = csv_files[0].stem
a = load_caravan_timeseries(CARAVAN_ROOT, check_id)
b = pd.read_csv(CSV_DIR / f"{check_id}.csv", parse_dates=["date"]).set_index("date")
b.loc[b[TARGET] < 0, TARGET] = np.nan
ok = np.allclose(a[DYNAMIC_INPUTS + [TARGET]].values, b[DYNAMIC_INPUTS + [TARGET]].values, atol=1e-3, equal_nan=True)
print(f"Kiem tra {check_id}: {a.shape}, khop voi CSV: {ok}")

## 4. Kiểm tra chất lượng và chọn lưu vực

Với mỗi lưu vực, ô này kiểm tra:

- **Số năm có nhãn** trong từng giai đoạn train / validation / test (lấy số ngày có lưu lượng chia 365,25).
- **Forcing có bị thiếu không** trong khoảng từ 364 ngày trước ngày bắt đầu train (để warm-up) đến hết tập test.

Lưu vực bị loại nếu không đủ `MIN_VALID_YEARS` ở **bất kỳ** giai đoạn nào, hoặc có forcing bị thiếu. Danh sách lưu vực được giữ lại ghi vào `result/e1/basins_e1.txt`.

In [ ]:
warmup_start = pd.Timestamp(PERIODS["train"][0]) - pd.Timedelta(days=SEQ_LENGTH - 1)
test_end = pd.Timestamp(PERIODS["test"][1])

rows = []
for nc in tqdm(sorted(NC_DIR.glob("*.nc")), desc="Kiem tra luu vuc"):
    with xr.open_dataset(nc) as ds:
        df = ds[DYNAMIC_INPUTS + [TARGET]].to_dataframe()
    rec = {"gauge_id": nc.stem}
    for p, (s, e) in PERIODS.items():
        rec[f"{p}_years"] = round(df.loc[s:e, TARGET].notna().sum() / 365.25, 2)
    rec["forcing_nan_days"] = int(df.loc[warmup_start:test_end, DYNAMIC_INPUTS].isna().any(axis=1).sum())
    rec["forcing_covers_period"] = bool(df.index.min() <= warmup_start and df.index.max() >= test_end)
    rows.append(rec)

qc = pd.DataFrame(rows).set_index("gauge_id")
reasons = []
for gid, r in qc.iterrows():
    why = [f"{p}<{MIN_VALID_YEARS[p]}nam" for p in PERIODS if r[f"{p}_years"] < MIN_VALID_YEARS[p]]
    if r.forcing_nan_days > 0: why.append(f"forcing thieu {r.forcing_nan_days} ngay")
    if not r.forcing_covers_period: why.append("forcing khong du dai")
    reasons.append("; ".join(why))
qc["exclude_reason"] = reasons
qc["selected"] = qc["exclude_reason"] == ""
qc.to_csv(E1_DIR / "basin_qc.csv")

selected = qc.index[qc.selected].tolist()
print(f"Tong: {len(qc)} luu vuc | Giu lai: {len(selected)} | Loai: {(~qc.selected).sum()}")
print(qc[[f"{p}_years" for p in PERIODS]].describe().round(2).loc[["min", "50%", "max"]])
if (~qc.selected).any():
    print("\nCac luu vuc bi loai:")
    print(qc.loc[~qc.selected, "exclude_reason"].to_string())

if QUICK_TEST:
    rng = np.random.default_rng(SEED)
    selected = sorted(rng.choice(selected, size=min(N_BASINS_QUICK, len(selected)), replace=False).tolist())
    print(f"\nQUICK_TEST: chi dung {len(selected)} luu vuc")

BASIN_FILE = E1_DIR / ("basins_e1_quick.txt" if QUICK_TEST else "basins_e1.txt")
BASIN_FILE.write_text("\n".join(selected))
print("Danh sach luu vuc:", BASIN_FILE)

## 5. Tạo file cấu hình NeuralHydrology

File YAML này khai báo toàn bộ pipeline: dữ liệu, cách chia, cửa sổ, model, loss, cách train và đánh giá. Chuẩn hóa (giống StandardScaler) được NeuralHydrology tự làm: mean/std tính **chỉ trên tập train** của tất cả lưu vực, rồi dùng lại cho val/test.

In [ ]:
fmt = lambda d: pd.Timestamp(d).strftime("%d/%m/%Y")
lr_yaml = "\n".join(f"  {k}: {v}" for k, v in LEARNING_RATE.items())
inputs_yaml = "\n".join(f"  - {v}" for v in DYNAMIC_INPUTS)
basin_path = BASIN_FILE.resolve().as_posix()

config_text = f"""# --- Thi nghiem ---
experiment_name: {EXPERIMENT}
run_dir: {(E1_DIR / 'runs').resolve().as_posix()}
seed: {SEED}
device: {DEVICE}

# --- Luu vuc va moc chia (A) ---
train_basin_file: {basin_path}
validation_basin_file: {basin_path}
test_basin_file: {basin_path}
train_start_date: {fmt(PERIODS['train'][0])}
train_end_date: {fmt(PERIODS['train'][1])}
validation_start_date: {fmt(PERIODS['validation'][0])}
validation_end_date: {fmt(PERIODS['validation'][1])}
test_start_date: {fmt(PERIODS['test'][0])}
test_end_date: {fmt(PERIODS['test'][1])}

# --- Du lieu ---
dataset: caravan
data_dir: {CARAVAN_ROOT.resolve().as_posix()}
dynamic_inputs:
{inputs_yaml}
target_variables:
  - {TARGET}
clip_targets_to_zero:
  - {TARGET}

# --- Cua so truot ---
seq_length: {SEQ_LENGTH}
predict_last_n: 1

# --- Model ---
model: cudalstm
head: regression
output_activation: linear
hidden_size: {HIDDEN_SIZE}
initial_forget_bias: 3
output_dropout: {OUTPUT_DROPOUT}

# --- Train ---
optimizer: Adam
loss: NSE
learning_rate:
{lr_yaml}
batch_size: {BATCH_SIZE}
epochs: {EPOCHS}
clip_gradient_norm: 1
num_workers: 0

# --- Danh gia ---
validate_every: 1
metrics:
  - NSE
  - KGE
  - Alpha-NSE
  - Beta-NSE
  - Pearson-r
  - FHV
  - FLV

# --- Log ---
log_interval: 5
log_tensorboard: False
log_n_figures: 0
save_weights_every: 1
"""
CONFIG_FILE = E1_DIR / f"{EXPERIMENT}.yml"
CONFIG_FILE.write_text(config_text, encoding="utf-8")
print(config_text)
print("Da ghi:", CONFIG_FILE)

## 6. Train

Mỗi epoch, NeuralHydrology train trên toàn bộ mẫu của tập train, rồi đánh giá trên tập validation và lưu trọng số. Kết quả nằm trong `result/e1/runs/<ten_thi_nghiem>_<ngay_gio>/`.

Thời gian ước tính trên RTX 5060 Ti với khoảng 480 lưu vực: vài phút đến khoảng 10 phút mỗi epoch, nên 30 epoch mất khoảng vài giờ. Bạn có thể xem tiến độ trong file `output.log` của thư mục run.

In [ ]:
from neuralhydrology.nh_run import start_run

runs_dir = E1_DIR / "runs"
before = set(runs_dir.glob(f"{EXPERIMENT}_*")) if runs_dir.exists() else set()
start_run(config_file=CONFIG_FILE)

new_runs = sorted(set(runs_dir.glob(f"{EXPERIMENT}_*")) - before)
RUN_DIR = new_runs[-1]
(E1_DIR / f"last_run_{EXPERIMENT}.txt").write_text(str(RUN_DIR.resolve()))
print("Run dir:", RUN_DIR)

## 7. Quá trình train và chọn epoch tốt nhất

Nếu kernel bị khởi động lại sau khi train xong, ô này tự đọc lại đường dẫn run cuối cùng. Bạn cũng có thể gán `RUN_DIR` bằng tay.

Epoch tốt nhất là epoch có **NSE trung vị trên tập validation cao nhất**. Tập test chỉ được dùng một lần, ở bước 8.

In [ ]:
import re
import matplotlib.pyplot as plt

# Màu và kiểu đường thống nhất cho mọi biểu đồ
C_BLUE, C_ORANGE, C_INK, C_MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ae", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "axes.labelcolor": C_MUTED, "xtick.color": C_MUTED, "ytick.color": C_MUTED,
    "axes.titlesize": 11, "lines.linewidth": 1.6, "legend.frameon": False,
})

if "RUN_DIR" not in globals():
    RUN_DIR = Path((E1_DIR / f"last_run_{EXPERIMENT}.txt").read_text().strip())
RUN_DIR = Path(RUN_DIR)
print("Run dir:", RUN_DIR)

num = r"(-?\d+(?:\.\d+)?(?:e-?\d+)?|nan)"
train_rows, val_rows = {}, {}
for line in (RUN_DIR / "output.log").read_text(encoding="utf-8").splitlines():
    m = re.search(rf"Epoch (\d+) average loss: .*?avg_total_loss: {num}", line)
    if m:
        train_rows[int(m.group(1))] = float(m.group(2))
    m = re.search(rf"Epoch (\d+) average validation loss: {num} -- Median validation metrics: (.*)", line)
    if m:
        metrics = dict(re.findall(rf"([\w-]+): {num}", m.group(3)))
        val_rows[int(m.group(1))] = {"val_loss": float(m.group(2)), **{k: float(v) for k, v in metrics.items()}}

hist = pd.DataFrame(val_rows).T.join(pd.Series(train_rows, name="train_loss"), how="outer").sort_index()
hist.index.name = "epoch"
nse_col = [c for c in hist.columns if c.endswith("NSE") and "Alpha" not in c and "Beta" not in c][0]
BEST_EPOCH = int(hist[nse_col].idxmax())
hist.to_csv(E1_DIR / f"{EXPERIMENT}_training_history.csv")
print(hist.round(4).to_string())
print(f"\nEpoch tot nhat (NSE trung vi val cao nhat): {BEST_EPOCH}  ->  NSE = {hist.loc[BEST_EPOCH, nse_col]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
ax.plot(hist.index, hist["train_loss"], color=C_BLUE, marker="o", ms=4, label="Train")
ax.plot(hist.index, hist["val_loss"], color=C_ORANGE, marker="o", ms=4, label="Validation")
ax.set_title("Loss NSE* (càng thấp càng tốt)"); ax.set_xlabel("Epoch"); ax.legend()
ax = axes[1]
ax.plot(hist.index, hist[nse_col], color=C_BLUE, marker="o", ms=4)
ax.scatter([BEST_EPOCH], [hist.loc[BEST_EPOCH, nse_col]], s=70, color=C_BLUE, edgecolor="white", zorder=3)
ax.annotate(f"tốt nhất: epoch {BEST_EPOCH}\nNSE = {hist.loc[BEST_EPOCH, nse_col]:.3f}",
            (BEST_EPOCH, hist.loc[BEST_EPOCH, nse_col]), textcoords="offset points", xytext=(8, -28),
            color=C_INK, fontsize=9)
ax.set_title("NSE trung vị trên tập validation"); ax.set_xlabel("Epoch")
for a in axes: a.xaxis.get_major_locator().set_params(integer=True)
plt.tight_layout(); plt.savefig(E1_DIR / f"{EXPERIMENT}_training_curves.png", dpi=150); plt.show()

## 8. Đánh giá trên tập test (01/10/1995 – 30/09/2010)

Dùng trọng số của epoch tốt nhất. Kết quả từng lưu vực được lưu vào `result/e1/<ten_thi_nghiem>_test_metrics.csv`.

Đường tham chiếu NSE = 0,63 là kết quả của "Global LSTM without static features" trong paper. Lưu ý: paper dùng forcing NLDAS và 531 lưu vực, còn ta dùng ERA5-Land, nên không kỳ vọng con số giống hệt.

In [ ]:
import pickle
from neuralhydrology.nh_run import eval_run

eval_run(run_dir=RUN_DIR, period="test", epoch=BEST_EPOCH)

test_dir = RUN_DIR / "test" / f"model_epoch{BEST_EPOCH:03d}"
metrics = pd.read_csv(test_dir / "test_metrics.csv", dtype={"basin": str}).set_index("basin")
metrics.to_csv(E1_DIR / f"{EXPERIMENT}_test_metrics.csv")
with open(test_dir / "test_results.p", "rb") as fp:
    TEST_RESULTS = pickle.load(fp)

nse = metrics.filter(regex=r"(^|_)NSE$").iloc[:, 0]
summary = pd.DataFrame({
    "trung_vi": metrics.median(numeric_only=True),
    "trung_binh": metrics.mean(numeric_only=True),
}).T
summary["n_luu_vuc"] = len(metrics)
summary["n_NSE_am"] = int((nse < 0).sum())
summary.to_csv(E1_DIR / f"{EXPERIMENT}_test_summary.csv")
print(summary.round(3).to_string())
print(f"\nNSE trung vi tap test: {nse.median():.3f}  (paper, LSTM khong co thuoc tinh tinh: 0.63)")

fig, ax = plt.subplots(figsize=(7, 4))
x = np.sort(nse.clip(lower=-1).values)
ax.step(x, np.arange(1, len(x) + 1) / len(x), where="post", color=C_BLUE, label="E1 (ERA5-Land, chỉ forcing)")
ax.axvline(0.63, color=C_MUTED, ls="--", lw=1)
ax.text(0.61, 0.97, "paper: 0,63", color=C_MUTED, fontsize=9, ha="right", va="top")
ax.axvline(nse.median(), color=C_BLUE, ls=":", lw=1)
ax.set_xlim(-1, 1); ax.set_ylim(0, 1)
ax.set_xlabel("NSE trên tập test (giá trị < -1 được gộp vào -1)"); ax.set_ylabel("Tỷ lệ lưu vực (tích lũy)")
ax.set_title(f"Phân bố NSE trên {len(nse)} lưu vực, trung vị = {nse.median():.3f}")
ax.legend(loc="upper left")
plt.tight_layout(); plt.savefig(E1_DIR / f"{EXPERIMENT}_test_nse_cdf.png", dpi=150); plt.show()

## 9. Hydrograph: lưu vực tốt nhất, trung vị và kém nhất

Mỗi biểu đồ so sánh lưu lượng quan trắc với lưu lượng mô phỏng trong một năm thủy văn của tập test. Đổi `WATER_YEAR` để xem năm khác (1996–2010).

In [ ]:
WATER_YEAR = 2005   # năm thủy văn 01/10/2004 -> 30/09/2005

def get_series(basin):
    xr_ = TEST_RESULTS[basin]["1D"]["xr"]
    obs = xr_[f"{TARGET}_obs"].isel(time_step=-1).to_series()
    sim = xr_[f"{TARGET}_sim"].isel(time_step=-1).to_series()
    # NeuralHydrology cắt giá trị mô phỏng âm về 0 khi tính metric (clip_targets_to_zero), nên ta cũng cắt khi vẽ
    return obs, sim.clip(lower=0)

ranked = nse.dropna().sort_values()
picks = {"Kém nhất": ranked.index[0], "Trung vị": ranked.index[len(ranked) // 2], "Tốt nhất": ranked.index[-1]}
start, end = f"{WATER_YEAR - 1}-10-01", f"{WATER_YEAR}-09-30"

fig, axes = plt.subplots(len(picks), 1, figsize=(11, 2.8 * len(picks)), sharex=False)
for ax, (label, basin) in zip(np.atleast_1d(axes), picks.items()):
    obs, sim = get_series(basin)
    ax.plot(obs.loc[start:end].index, obs.loc[start:end].values, color=C_BLUE, label="Quan trắc")
    ax.plot(sim.loc[start:end].index, sim.loc[start:end].values, color=C_ORANGE, label="Mô phỏng")
    ax.set_title(f"{label}: {basin}  (NSE cả tập test = {nse[basin]:.3f})", loc="left")
    ax.set_ylabel("mm/ngày")
    ax.legend(loc="upper right", ncol=2)
plt.tight_layout(); plt.savefig(E1_DIR / f"{EXPERIMENT}_hydrographs_WY{WATER_YEAR}.png", dpi=150); plt.show()

## Tiếp theo

- Nếu đang ở `QUICK_TEST = True` và mọi ô chạy không lỗi: đổi thành `False` ở ô 1, rồi chạy lại từ ô 1 (bỏ qua ô 3).
- Sau khi có kết quả 1 seed: chạy thêm 9 seed khác (đổi `SEED`) để làm ensemble 10 model như paper.
- Các file kết quả chính trong `result/e1/`: `basin_qc.csv`, `*_training_history.csv`, `*_test_metrics.csv`, `*_test_summary.csv` và các hình `.png`.